# 📊 Naive Bayes Comparison Notebook
So sánh Custom vs Sklearn Naive Bayes trên tập digits.

## 1. Import thư viện

In [ ]:
import numpy as npimport timefrom sklearn.datasets import load_digitsfrom sklearn.model_selection import train_test_splitfrom sklearn.naive_bayes import MultinomialNB, BernoulliNBfrom sklearn.metrics import classification_report, confusion_matrix

## 2. Load và chia dữ liệu

In [ ]:
data = load_digits()X = data.datay = data.targetX_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42)print("Data ready!")

## 3. Custom Multinomial Naive Bayes

In [ ]:
class CustomMultinomialNB:    def __init__(self, alpha=1.0):        self.alpha = alpha    def fit(self, X, y):        self.classes = np.unique(y)        self.class_log_prior = []        self.feature_log_prob = []        for c in self.classes:            X_c = X[y == c]            prior = X_c.shape[0] / X.shape[0]            self.class_log_prior.append(np.log(prior))            feature_count = X_c.sum(axis=0) + self.alpha            total_count = feature_count.sum()            prob = feature_count / total_count            self.feature_log_prob.append(np.log(prob))        self.class_log_prior = np.array(self.class_log_prior)        self.feature_log_prob = np.array(self.feature_log_prob)    def predict(self, X):        log_probs = X @ self.feature_log_prob.T + self.class_log_prior        return self.classes[np.argmax(log_probs, axis=1)]

## 4. Custom Bernoulli Naive Bayes

In [ ]:
class CustomBernoulliNB:    def __init__(self, alpha=1.0):        self.alpha = alpha    def fit(self, X, y):        X = (X > 0).astype(int)        self.classes = np.unique(y)        self.class_log_prior = []        self.feature_log_prob = []        for c in self.classes:            X_c = X[y == c]            prior = X_c.shape[0] / X.shape[0]            self.class_log_prior.append(np.log(prior))            prob = (X_c.sum(axis=0) + self.alpha) / (X_c.shape[0] + 2*self.alpha)            self.feature_log_prob.append(np.log(prob))        self.class_log_prior = np.array(self.class_log_prior)        self.feature_log_prob = np.array(self.feature_log_prob)    def predict(self, X):        X = (X > 0).astype(int)        log_probs = X @ self.feature_log_prob.T + self.class_log_prior        return self.classes[np.argmax(log_probs, axis=1)]

## 5. Khởi tạo model

In [ ]:
models = {    "Custom Multinomial NB": CustomMultinomialNB(),    "Custom Bernoulli NB": CustomBernoulliNB(),    "Sklearn Multinomial NB": MultinomialNB(),    "Sklearn Bernoulli NB": BernoulliNB()}train_time = {}results = {}

## 6. Train model

In [ ]:
for name, model in models.items():    start = time.time()    model.fit(X_train, y_train)    train_time[name] = time.time() - start

## 7. Đánh giá

In [ ]:
for name, model in models.items():    y_pred = model.predict(X_test)    acc = np.mean(y_pred == y_test)    results[name] = acc    print("\n", name)    print("Accuracy:", round(acc, 4))    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))    print("Classification Report:\n", classification_report(y_test, y_pred, zero_division=0))

## 8. So sánh kết quả

In [ ]:
print("\n===== COMPARISON =====")for name in models:    print(f"{name:30} | Acc: {results[name]:.4f} | Time: {train_time[name]:.4f}s")

## 9. Model tốt nhất

In [ ]:
best_model = max(results, key=results.get)print("Best model:", best_model)

## 10. Bảng kết quả

In [ ]:
import pandas as pddf = pd.DataFrame({    "Model": list(results.keys()),    "Accuracy": list(results.values()),    "Training Time (s)": list(train_time.values())})display(df)

## 11. Visualization

In [ ]:
import matplotlib.pyplot as pltplt.figure()plt.bar(df["Model"], df["Accuracy"])plt.xticks(rotation=45)plt.title("Accuracy Comparison")plt.show()plt.figure()plt.bar(df["Model"], df["Training Time (s)"])plt.xticks(rotation=45)plt.title("Training Time Comparison")plt.show()